In [5]:
# -------------------------------------------------------
# Flavor Profile Model - fully self-contained
# -------------------------------------------------------

import re
import joblib
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import NMF
from sklearn.preprocessing import MultiLabelBinarizer

try:
    from spice_data_v2 import SPICES, ALIASES, CANONICAL_SPICES, FLAVOR_PROFILES, REGION_PROFILES
except ModuleNotFoundError:
    print("spice_data_v2 not found. Using fallback spice definitions.")
    SPICES = [
        "salt", "pepper", "cumin", "paprika", "garlic", "onion",
        "coriander", "turmeric", "ginger", "chili", "oregano", "thyme"
    ]
    ALIASES = {s: s for s in SPICES}
    CANONICAL_SPICES = sorted(set(SPICES))
    FLAVOR_PROFILES = {}
    REGION_PROFILES = {}

# --- config ---
CSV_PATH     = "/Users/daniellarson/Desktop/SpiceRack/cookingdataset/RecipeNLG_dataset.csv"
MODEL_PATH   = "flavor_profile_model.joblib"
SAMPLE_SIZE  = None
N_COMPONENTS = 20
NMF_SAMPLE   = 200_000

# --- helpers ---
def clean(s):
    if not isinstance(s, str): return ""
    s = s.lower()
    s = re.sub(r"[^a-z\s']", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def to_canonical(spice):
    n = clean(spice)
    return clean(ALIASES.get(n, n))

SPICE_PATTERNS = sorted(
    [(sp, clean(sp), re.compile(rf"(^| ){re.escape(clean(sp))}( |$)")) for sp in SPICES],
    key=lambda x: -len(x[0])
)

def get_spices_from_recipe(ingredients):
    if isinstance(ingredients, list):
        raw = " ".join(str(x) for x in ingredients)
    else:
        raw = str(ingredients)
    text = clean(raw)
    found = set()
    for _, norm, pat in SPICE_PATTERNS:
        if pat.search(" " + text + " "):
            found.add(norm)
    return {to_canonical(sp) for sp in found}

def parse_ingredient_string(x):
    if isinstance(x, list): return [str(i) for i in x]
    if not isinstance(x, str): return []
    s = x.strip()
    if s.startswith("[") and s.endswith("]"):
        items = re.findall(r"'([^']*)'|\"([^\"]*)\"|", s)
        parsed = [a if a else b for a, b in items]
        return parsed if parsed else [s]
    return [s]

# --- load data ---
print(f"loading {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)
if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

df["ingredients_parsed"] = df["ingredients"].apply(parse_ingredient_string)
df["spices"]             = df["ingredients_parsed"].apply(get_spices_from_recipe)

mlb = MultiLabelBinarizer(classes=sorted(CANONICAL_SPICES))
X   = np.asarray(mlb.fit_transform(df["spices"]))
print(f"loaded {len(df):,} recipes — matrix shape: {X.shape}")

# --- train or load model ---
if Path(MODEL_PATH).exists():
    print("found saved model, loading...")
    saved       = joblib.load(MODEL_PATH)
    nmf         = saved["nmf"]
    spice_names = saved["classes"]
else:
    print("no saved model found, training...")
    train_X = X[:NMF_SAMPLE] if NMF_SAMPLE and NMF_SAMPLE < len(X) else X
    n_samples, n_features = train_X.shape
    requested_components = N_COMPONENTS
    n_components = max(1, min(requested_components, n_samples, n_features))
    init_method = "nndsvda" if n_components <= min(n_samples, n_features) else "random"
    if requested_components != n_components:
        print(f"Adjusted n_components from {requested_components} to {n_components} (n_samples={n_samples}, n_features={n_features})")
    if init_method != "nndsvda":
        print(f"Using fallback NMF init='{init_method}'")
    t0  = time.time()
    nmf = NMF(n_components=n_components, init=init_method, max_iter=300, random_state=42)
    nmf.fit(train_X)
    spice_names = list(mlb.classes_)
    joblib.dump({"nmf": nmf, "classes": spice_names}, MODEL_PATH)
    print(f"trained in {time.time()-t0:.1f}s — saved to {MODEL_PATH}")

H = nmf.components_

LEARNED_LABELS = []
for row in H:
    top_idx = row.argsort()[-3:][::-1]
    LEARNED_LABELS.append(" / ".join(spice_names[i] for i in top_idx))

def get_learned_profiles(pantry, top_k=5, min_score=0.05):
    vec         = np.asarray(mlb.transform([pantry]), dtype=float)
    scores      = (vec @ H.T).flatten()
    max_s       = H.sum(axis=1)
    max_s[max_s == 0] = 1
    norm_scores = scores / max_s

    results = []
    for i in norm_scores.argsort()[::-1][:top_k]:
        s = float(norm_scores[i])
        if s < min_score:
            break
        pantry_idx     = [spice_names.index(sp) for sp in pantry if sp in spice_names]
        driving_spices = sorted(pantry_idx, key=lambda j: -H[i][j])[:5]
        results.append((LEARNED_LABELS[i], round(s, 3), [spice_names[j] for j in driving_spices]))

    return results

# n_components may not be recognized by static checkers as a direct attribute on sklearn NMF
components = nmf.get_params().get("n_components", N_COMPONENTS)
print(f"ready — {components} components, {len(spice_names)} spices")

spice_data_v2 not found. Using fallback spice definitions.
loading /Users/daniellarson/Desktop/SpiceRack/cookingdataset/RecipeNLG_dataset.csv...
loaded 2,231,142 recipes — matrix shape: (2231142, 12)
no saved model found, training...
Adjusted n_components from 20 to 12 (n_samples=200000, n_features=12)
trained in 4.0s — saved to flavor_profile_model.joblib
ready — 12 components, 12 spices
